In [30]:
from pathlib import Path
import polars as pl

"""
Query collector-peer percentile-bin assignments.
    This notebook demonstrates how to retrieve collector-peer pairs
    belonging to a selected percentile bin on a selected day.
    Modify DATE and BIN below to explore the dataset.
"""

DATA = Path("/noisy-neighbors/data/collector_peer_daily_bins.parquet")

DATE = "2012-01-01"
BIN = "95-100%"
LIMIT = 20

VALID_BINS = {"95-100%", "75-95%", "50-75%", "Lower 50%"}

if BIN not in VALID_BINS:
    raise ValueError(f"Invalid BIN={BIN}. Expected one of: {sorted(VALID_BINS)}")

if not DATA.exists():
    raise FileNotFoundError(f"Input file not found: {DATA}")

data = pl.scan_parquet(DATA)
date_filter = pl.lit(DATE).str.strptime(pl.Date, "%Y-%m-%d")
collector_peer_pairs = (
    data.filter(
        (pl.col("date") == date_filter)
        & (pl.col("percentile_bin") == BIN))
        .select("date","collector","peer_asn","update_count",
        "daily_rank","percentile_bin",).sort("daily_rank").collect())
collector_peer_pairs

date,collector,peer_asn,update_count,daily_rank,percentile_bin
datetime[ns],str,i64,i64,i64,str
2012-01-01 00:00:00,"""routeviews.sao…",28289,1991868,1,"""95-100%"""
2012-01-01 00:00:00,"""routeviews.lin…",6667,1496544,2,"""95-100%"""
2012-01-01 00:00:00,"""routeviews.rou…",3549,1406870,3,"""95-100%"""
2012-01-01 00:00:00,"""routeviews.lin…",3257,1109145,4,"""95-100%"""
2012-01-01 00:00:00,"""routeviews.lin…",13237,896805,5,"""95-100%"""
2012-01-01 00:00:00,"""routeviews.lin…",13030,873780,6,"""95-100%"""


In [ ]:
"""
/noisy-neighbors/data/coollector_peer_bins/
    p95_100.parquet:
        Daily collector-peer pairs belonging to the top 5%
        of update volume on each day.
    p75_95.parquet:
        Daily collector-peer pairs belonging to the 75–95%
        update-volume range on each day.
    p50_75.parquet:
        Daily collector-peer pairs belonging to the 50–75%
        update-volume range on each day.
    p00_50.parquet:
        Daily collector-peer pairs belonging to the lower 50%
        of update volume on each day.
"""

DATA = Path("/noisy-neighbors/data/collector_peer_daily_bins.parquet")
OUT_DIR = Path("/noisy-neighbors/data/coollector_peer_bins")
OUT_DIR.mkdir(parents=True, exist_ok=True)
BIN_FILES = {"95-100%": "p95_100.parquet","75-95%": "p75_95.parquet",
             "50-75%": "p50_75.parquet","Lower 50%": "p00_50.parquet",}

for bin_name, outfile in BIN_FILES.items():
    (
        pl.scan_parquet(DATA)
        .filter(pl.col("percentile_bin") == bin_name)
        .select(
            "date",
            "collector",
            "peer_asn",
            "update_count",
            "daily_rank",
            "daily_percentile",
        )
        .sink_parquet(OUT_DIR / outfile))
print(f"Saved: {OUT_DIR}")

Saved: /home/ebrima/disk2/noisy-neighbors/data/coollector_peer_bins
